In [66]:
import os

import scripts.helpers as helpers
helpers.add_backend_to_path()

import app.courses as courses

cs_client= courses.CourseClient(os.path.join("..", "web", "backend", "assets", "courses"))

data = cs_client.all_courses()

print(f'Loaded {len(data)} courses')

Loaded 21106 courses


In [89]:
def preprocess_prerequisites(prerequisites) -> list[str]:
    if not prerequisites or prerequisites == "":
        return []
    if not isinstance(prerequisites, str):
        raise ValueError(f"Prerequisites must be a string, got {type(prerequisites)}")

    import re
    return [re.sub(r"předp\.", "", prerequisites, flags=re.IGNORECASE).strip()]

def tokenize_prerequisites(prerequisites: str) -> list[str]:
    if prerequisites is None:
        return []
    if not isinstance(prerequisites, str):
        raise ValueError(f"Prerequisites must be a string, got {type(prerequisites)}")

    s = prerequisites.strip()
    if not s:
        return []

    tokens: list[str] = []
    i = 0
    n = len(s)

    while i < n:
        ch = s[i]

        # skip whitespace
        if ch.isspace():
            i += 1
            continue

        # parentheses
        if ch == '(' or ch == ')':
            tokens.append(ch)
            i += 1
            continue

        # double-char logical operators
        if ch == '&' and i + 1 < n and s[i + 1] == '&':
            tokens.append('&&')
            i += 2
            continue
        if ch == '|' and i + 1 < n and s[i + 1] == '|':
            tokens.append('||')
            i += 2
            continue

        # negation / not-equal
        if ch == '!':
            if i + 1 < n and s[i + 1] == '=':
                tokens.append('!=')
                i += 2
            else:
                tokens.append('!')
                i += 1
            continue

        # separators kept as individual tokens
        if ch == '-' or ch == ',':
            tokens.append(ch)
            i += 1
            continue

        # numeric literal
        if ch.isdigit():
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            tokens.append(s[i:j])
            i = j
            continue

        # identifier / course code (letters, digits, underscore)
        if ch.isalpha() or ch == '_' or ch.isdigit():
            j = i + 1
            while j < n and (s[j].isalnum() or s[j] == '_'):
                j += 1
            tokens.append(s[i:j])
            i = j
            continue

        # fallback: keep single char so nothing is lost
        tokens.append(ch)
        i += 1

    return tokens

def remove_functions(tokens: list[str], functions: list[str]) -> list[str]:
    found_functions: list[tuple[int, int]] = []
    function_start: int | None = None
    nested_count = 0
    functions_lower = [f.lower() for f in functions]
    
    for i, token in enumerate(tokens):
        if function_start is None and token.lower() in functions_lower:
            function_start = i - 1 if i > 0 and tokens[i - 1] == '!' else i
            nested_count = 0 
            
        if function_start is None:
            continue
        
        if token == '(':
            nested_count += 1
        elif token == ')':
            nested_count -= 1
            if nested_count == 0:
                found_functions.append((function_start, i))
                function_start = None
    
    for start, end in reversed(found_functions):
        del tokens[start:end + 1]
    
    return tokens

def remove_souhlas(tokens: list[str]) -> list[str]:
    return [t for t in tokens if t.lower() != 'souhlas']

def cleanup_brackets(tokens: list[str]) -> (list[str], bool):
    if not tokens:
        return [], False

    t = tokens[:]
    changed_outer = False   

    while True:
        changed = False

        # Remove empty parentheses: ()
        i = 0
        while i < len(t) - 1:
            if t[i] == '(' and t[i + 1] == ')':
                del t[i:i + 2]
                changed = True
            else:
                i += 1
        
        # Remove outer parentheses
        if t and t[0] == "(" and t[-1] == ")":
            are_outer = True
            depth = 0
            for i in range(1, len(t) - 1):
                if t[i] == "(":
                    depth += 1
                elif t[i] == ")":
                    depth -= 1
                    if depth < 0:
                        are_outer = False
                        break
            if are_outer:
                del t[0]
                t.pop()
                changed = True
        
        # Remove redundant inner parentheses
        i = 0
        paren_start: int | None = None
        while i < len(t):
            if paren_start is not None and (t[i] == "&&" or t[i] == "||"):
                paren_start = None
            if t[i] == "(":
                paren_start = i
            elif t[i] == ")" and paren_start is not None:
                del t[i]
                del t[paren_start]
                changed = True
                paren_start = None
                i -= 1
                continue
            i += 1

        if changed: 
            changed_outer = True
        else:
            break
        
    return t, changed_outer

def cleanup_logical_operators(tokens: list[str]) -> (list[str], bool):
    if not tokens:
        return [], False

    t = tokens[:]
    changed_outer = False
    logical_ops = {"&&", "||"}

    while True:
        changed = False

        # Remove boundary operators
        while t and t[0] in logical_ops:
            del t[0]
            changed = True
        while t and t[-1] in logical_ops:
            t.pop()
            changed = True

        # Remove duplicate and mixed logical operators
        i = 0
        while i < len(t) - 1:
            a, b = t[i], t[i+1]
            if a in logical_ops and b in logical_ops:
                if a == b:
                    del t[i]
                else:
                    del t[i+1 if a == "||" else i]
                changed = True
            else:
                i += 1
        
        # Remove logical operators that are not surrounded by parentheses
        i = 0
        while i < len(t) - 1:
            if t[i] in logical_ops and (t[i-1] == "(" or t[i+1] == ")"):
                del t[i]
                changed = True
            else:
                i += 1

        if changed:
            changed_outer = True
        else:
            break

    return t, changed_outer

def cleanup_tokens(tokens: list[str]) -> list[str]:
    while True:
        tokens, changed_outer_logical = cleanup_logical_operators(tokens)
        tokens, changed_outer_brackets = cleanup_brackets(tokens)

        if not changed_outer_brackets and not changed_outer_logical:
            break
    return tokens

def get_functions_from_tokens(tokens: list[str]) -> list[str]:
    functions = []
    for i, token in enumerate(tokens):
        if token in ('==', '!=', '&&', '||', '>', '<', '>=', '<=', '(', ')', '!'):
            continue

        if i == len(tokens) - 1 or tokens[i + 1] != '(':
            continue

        functions.append(token)
    return functions

def remove_negation(tokens: list[str]) -> list[str]:
    i = 0
    while i < len(tokens) - 1:
        if tokens[i] == '!' and tokens[i + 1] != '(':
            del tokens[i]
            del tokens[i]
        else:
            i += 1
    return tokens

"""
# Get all functions from all courses

all_functions: dict[str, int] = {}
courses_with_functions = 0

for course in data:
    processed_prerequisites = preprocess_prerequisites(course.PREREQUISITES)
    if not processed_prerequisites or len(processed_prerequisites) == 0:
        continue

    tokens = tokenize_prerequisites(processed_prerequisites[0])

    tokens = remove_functions(tokens, [
        'nowany', 'now', 'studijni_skupina', 'kredity_min', 'bozp_ok', 'vycet', 'forma',
        'program', 'fakulta', 'typ_studia', 'rocnik', 'semestr', 'obor'
    ])

    has_function = False
    functions = get_functions_from_tokens(tokens)
    for function in functions:
        if function.lower() in all_functions:
            all_functions[function.lower()] += 1
        else:
            all_functions[function.lower()] = 1

    if functions:
        courses_with_functions += 1
        print(f"{course.CODE:<10} | Prerequisites: {tokens}")

print("All functions: ", all_functions)
print(f"Courses with functions: {courses_with_functions}")
print(f"Found {len(all_functions)} functions")
"""
course_prerequisites: map[str, list[str]] = {}

with_prerequisites = 0
for course in data:
    processed_prerequisites = preprocess_prerequisites(course.PREREQUISITES)
    if not processed_prerequisites or len(processed_prerequisites) == 0:
        continue

    tokens = tokenize_prerequisites(processed_prerequisites[0])

    original_tokens = tokens.copy()

    tokens = remove_functions(tokens, [
        'nowany', 'now', 'studijni_skupina', 'kredity_min', 'bozp_ok', 'vycet', 'forma',
        'program', 'fakulta', 'typ_studia', 'rocnik', 'semestr', 'obor'
    ])
    tokens = remove_souhlas(tokens)

    tokens = remove_negation(tokens)

    tokens = cleanup_tokens(tokens)

    if tokens:
        course_prerequisites[course.CODE] = [token for token in tokens if token not in ('&&', '||', '(', ')', '!', course.CODE)]
        with_prerequisites += 1
        # print(f"{course.CODE:<10} | Prerequisites: {tokens}; Original: {original_tokens}")
        print(f"{course.CODE:<10} | Prerequisites: {tokens}")

print(f"Courses with prerequisites: {with_prerequisites}")
print(course_prerequisites)

RSCZJ01    | Prerequisites: ['SPCZJ02', '||', 'SPCZJ02Z']
SPAJK001   | Prerequisites: ['SPAJ003']
SPCZJ02    | Prerequisites: ['SPCZJ01']
BKF_BAS1   | Prerequisites: ['BKF_TEBP']
BKF_BAS2   | Prerequisites: ['BKF_BAS1']
BKF_FIU2   | Prerequisites: ['BKF_FIU1', '||', 'BKF_FIC1']
BKF_SZP1   | Prerequisites: ['BKF_TEZP']
BKF_SZP2   | Prerequisites: ['BKF_SZP1']
BKH_BAS1   | Prerequisites: ['BKH_TEBP']
BKH_BAS2   | Prerequisites: ['BKH_BAS1']
BKH_DIMA   | Prerequisites: ['BKH_ZAMO', '||', 'BKR_MACR']
BKH_SZP1   | Prerequisites: ['BKH_TEZP']
BKH_SZP2   | Prerequisites: ['BKH_SZP1']
BKJ_JZA1   | Prerequisites: ['BKJ_VTJA']
BKJ_JZA2   | Prerequisites: ['BKJ_JZA1']
BKM_APS2   | Prerequisites: ['BKM_APS1']
BKM_SZP1   | Prerequisites: ['BKM_TEZP']
BKM_SZP2   | Prerequisites: ['BKM_SZP1']
BKR_SZP1   | Prerequisites: ['BKR_TEZP']
BKR_SZP2   | Prerequisites: ['BKR_SZP1']
BKV_BAS1   | Prerequisites: ['BKV_TEBP']
BKV_BAS2   | Prerequisites: ['BKV_BAS1']
BKV_VEF1   | Prerequisites: ['BKE_MAE1']
BPE_BA

In [90]:
from collections import defaultdict
import networkx as nx  # optional


def build_prereq_dependency_graph(course_prereqs: dict[str, list[str]]):
    # Map each prerequisite to the list of courses that require it
    prereq_to_dependents: dict[str, set[str]] = defaultdict(set)

    for course_code, prereqs in course_prereqs.items():
        for prereq in prereqs:
            # Safety check in case any logical tokens slipped through
            if prereq in {"&&", "||", "(", ")", "!"}:
                continue
            prereq_to_dependents[prereq].add(course_code)

    # Build a directed graph: prerequisite -> dependent course
    graph = None
    if nx is not None:
        graph = nx.DiGraph()
        for prereq, dependents in prereq_to_dependents.items():
            for dependent in dependents:
                graph.add_edge(prereq, dependent)

    return prereq_to_dependents, graph

prereq_to_dependents, prereq_graph = build_prereq_dependency_graph(course_prerequisites)

num_nodes = (len(prereq_to_dependents)
             + len({c for deps in prereq_to_dependents.values() for c in deps}))
num_edges = sum(len(deps) for deps in prereq_to_dependents.values())

print(f"Prereq->Dependents map: {len(prereq_to_dependents)} prerequisites")
print(f"Graph (if nx available): nodes≈{num_nodes}, edges={num_edges}")

# Example: show dependents for a specific prerequisite code
example_prereq = next(iter(prereq_to_dependents)) if prereq_to_dependents else None
if example_prereq:
    print(f"Example: {example_prereq} → {sorted(list(prereq_to_dependents[example_prereq]))[:10]}")

# Graph stats
if nx is not None and prereq_graph is not None:
    print("\n--- Graph stats ---")

    # 1) Degrees and top nodes
    in_degrees = dict(prereq_graph.in_degree())   # number of prerequisites each course has (incoming edges)
    out_degrees = dict(prereq_graph.out_degree()) # number of dependents each prereq has (outgoing edges)

    if out_degrees:
        top_prereq, top_dependents = max(out_degrees.items(), key=lambda kv: kv[1])
        print(f"Most dependents: {top_prereq} → {top_dependents} courses")
    if in_degrees:
        top_course, top_prereqs = max(in_degrees.items(), key=lambda kv: kv[1])
        print(f"Most prerequisites: {top_course} has {top_prereqs}")

    sources = [n for n, d in in_degrees.items() if d == 0]
    sinks = [n for n, d in out_degrees.items() if d == 0]
    print(f"Sources (no prerequisites): {len(sources)}")
    print(f"Sinks (no dependents): {len(sinks)}")

    # 2) Cycles and SCCs
    sccs = list(nx.strongly_connected_components(prereq_graph))
    cyclic_sccs = [c for c in sccs if len(c) > 1]
    print(f"Strongly connected components: {len(sccs)} (cyclic: {len(cyclic_sccs)})")
    if cyclic_sccs:
        # Show up to 3 sample cycles by extracting simple cycles
        simple_cycles = list(nx.simple_cycles(prereq_graph))
        print(f"Detected simple cycles: {len(simple_cycles)}")
        for i, cyc in enumerate(simple_cycles[:3]):
            print(f"  cycle[{i+1}]: {' → '.join(cyc + [cyc[0]])}")

    # 3) Longest chain length using condensation DAG
    # Collapse each SCC into a single node to ensure DAG, then compute longest path length
    cdag = nx.condensation(prereq_graph)
    # Longest path in DAG via DP
    longest_len = 0
    longest_end = None
    topo = list(nx.topological_sort(cdag))
    dist = {n: 0 for n in topo}
    pred = {n: None for n in topo}
    for u in topo:
        for v in cdag.successors(u):
            if dist[u] + 1 > dist[v]:
                dist[v] = dist[u] + 1
                pred[v] = u
                if dist[v] > longest_len:
                    longest_len = dist[v]
                    longest_end = v

    # Reconstruct one longest path at SCC-level
    scc_path = []
    cur = longest_end
    while cur is not None:
        scc_path.append(cur)
        cur = pred[cur]
    scc_path.reverse()

    # Map SCC indices back to original nodes; show a representative node per SCC
    rep_nodes = []
    comp = cdag.graph['mapping']  # original node -> scc index
    inv_comp = defaultdict(list)
    for node, cid in comp.items():
        inv_comp[cid].append(node)
    for cid in scc_path:
        # pick the first node as representative, or join a few if cyclic SCC
        members = inv_comp.get(cid, [])
        rep = ",".join(members[:3]) if members else str(cid)
        rep_nodes.append(rep)

    print(f"Longest chain length (SCC-collapsed edges): {longest_len}")
    if rep_nodes:
        print(f"Longest chain (representatives): {' → '.join(rep_nodes)}")
else:
    print("NetworkX not available; skipping graph stats.")

Prereq->Dependents map: 4246 prerequisites
Graph (if nx available): nodes≈9105, edges=11583
Example: SPCZJ02 → ['RSCZJ01']

--- Graph stats ---
Most dependents: pov_tv → 76 courses
Most prerequisites: VLPG11XX has 44
Sources (no prerequisites): 2354
Sinks (no dependents): 2967
Strongly connected components: 7200 (cyclic: 7)
Detected simple cycles: 31
  cycle[1]: G8021k → G8021 → G8021k
  cycle[2]: Bi1090c → Bi1090 → Bi1090c
  cycle[3]: GA021k → GA041 → GA021k
Longest chain length (SCC-collapsed edges): 16
Longest chain (representatives): ZD011 → ZD012 → ZD013 → ZD014 → ZD015 → ZD016 → ZD017 → ZD018 → ZD019 → ZD020 → ZD028 → ZD030 → ZD033 → ZD034 → ZD035 → ZD036 → ZD037
